# SpaceX Falcon 9 — Interactive Visual Analytics with Folium

Builds three interactive Leaflet/Folium maps: (1) all launch site locations,
(2) launch outcomes color-coded green/red on a `MarkerCluster`, and (3) the
proximity of a selected pad (CCAFS SLC-40) to the coastline, a highway and a
railway, with `PolyLine` distances calculated via the haversine formula.

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from folium.features import DivIcon
import math

BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"
geo = pd.read_csv(f"{BASE}/datasets/spacex_launch_geo.csv")
sites = geo.groupby('Launch Site')[['Lat', 'Long']].first().reset_index()
sites

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [2]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6373
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

## Map 1 — All launch site locations

In [3]:
site_map = folium.Map(location=[28.5, -80.6], zoom_start=4.4)
for _, r in sites.iterrows():
    folium.Circle([r.Lat, r.Long], radius=1000, color='#0F62FE', fill=True,
                  fill_opacity=0.5, popup=r['Launch Site']).add_to(site_map)
    folium.map.Marker([r.Lat, r.Long], icon=DivIcon(icon_size=(220, 24), icon_anchor=(0, 0),
        html=f'<div style="font-size:13px;font-weight:700;color:#002D9C;">{r["Launch Site"]}</div>')).add_to(site_map)
site_map.save('map1_all_sites.html')
site_map

## Map 2 — Launch outcomes colored green (success) / red (failure)

In [4]:
outcome_map = folium.Map(location=[28.56, -80.60], zoom_start=9)
marker_cluster = MarkerCluster().add_to(outcome_map)
for _, r in geo.iterrows():
    color = 'green' if r['class'] == 1 else 'red'
    folium.Marker([r['Lat'], r['Long']], icon=folium.Icon(color=color, icon='rocket', prefix='fa'),
                  popup=f"{r['Launch Site']} — {r['Booster Version']} — {'Success' if r['class']==1 else 'Failure'}"
                  ).add_to(marker_cluster)
outcome_map.save('map2_outcomes.html')
outcome_map

## Map 3 — CCAFS SLC-40 proximity to coastline / highway / railway

In [5]:
launch_site = sites[sites['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
site_lat, site_lon = launch_site.Lat, launch_site.Long
coastline, highway, railway = (28.56335, -80.56779), (28.56321, -80.57083), (28.57419, -80.58611)

proximity_map = folium.Map(location=[site_lat, site_lon], zoom_start=14)
folium.Circle([site_lat, site_lon], radius=600, color='#0F62FE', fill=True, fill_opacity=0.4).add_to(proximity_map)

for label, coord, color in [('Coastline', coastline, '#1192E8'), ('Highway (Phillips Pkwy)', highway, '#FF832B'), ('Railway', railway, '#8A3FFC')]:
    dist = haversine(site_lat, site_lon, coord[0], coord[1])
    folium.Marker(coord, popup=f"{label}: {dist:.2f} km").add_to(proximity_map)
    folium.PolyLine([[site_lat, site_lon], coord], color=color, weight=3, dash_array='8').add_to(proximity_map)
    print(f"{label}: {dist:.2f} km from CCAFS SLC-40")

proximity_map.save('map3_proximity.html')
proximity_map

Coastline: 0.88 km from CCAFS SLC-40
Highway (Phillips Pkwy): 0.59 km from CCAFS SLC-40
Railway: 1.52 km from CCAFS SLC-40
